# 1. Installation & Setup

In [ ]:
!pip install pyarabic bs4 torch tqdm

In [ ]:
import os
import re
import pickle
import torch
import torch.nn as nn
import pyarabic.araby as araby
from bs4 import BeautifulSoup
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

# Constants for T4 GPU efficiency
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 256  # Sequence length for character-level modeling
BATCH_SIZE = 32
D_MODEL = 512
N_HEAD = 8
N_LAYERS = 8  # High capacity for Classical Arabic accuracy

# 2. Data Preprocessing & Extraction

In [ ]:
# Define the set of unique diacritic combinations
DIACRITICS = [
    '', araby.FATHA, araby.DAMMA, araby.KASRA, araby.SUKUN,
    araby.FATHATAN, araby.DAMMATAN, araby.KASRATAN,
    araby.SHADDA,
    araby.SHADDA + araby.FATHA, araby.SHADDA + araby.DAMMA,
    araby.SHADDA + araby.KASRA, araby.SHADDA + araby.FATHATAN,
    araby.SHADDA + araby.DAMMATAN, araby.SHADDA + araby.KASRATAN
]
class_to_id = {d: i for i, d in enumerate(DIACRITICS)}
id_to_class = {i: d for d, i in class_to_id.items()}

In [ ]:
def extract_raw_text(folder_path):
    """Parses .htm files and extracts diacritized Arabic sentences."""
    sentences = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".htm"):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f, 'html.parser')
                text = soup.get_text()
                # Split by Arabic punctuation
                raw_sentences = re.split(r'[.\u061f!:]', text)
                for s in raw_sentences:
                    clean = s.strip()
                    if 20 < len(clean) < MAX_LEN:
                        sentences.append(clean)
    return sentences

In [ ]:
def extract_char_diacritics(text):
    """Extract (base_char, diacritics) pairs from diacritized Arabic text."""
    result = []
    i = 0
    while i < len(text):
        if not araby.is_tashkeel(text[i]):
            base_char = text[i]
            diacritics = ''
            j = i + 1
            while j < len(text) and araby.is_tashkeel(text[j]):
                diacritics += text[j]
                j += 1
            result.append((base_char, diacritics))
            i = j
        else:
            i += 1
    return result


class DiacritizationDataset(Dataset):
    def __init__(self, sentences, char_to_id):
        self.data = []
        for sent in sentences:
            pairs = extract_char_diacritics(sent)
            input_ids = [char_to_id.get(c, char_to_id['<UNK>']) for c, _ in pairs]
            label_ids = [class_to_id.get(d, 0) for _, d in pairs]
            self.data.append((torch.tensor(input_ids), torch.tensor(label_ids)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    inputs, labels = zip(*batch)
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=-1)
    return inputs_padded, labels_padded

# 3. Transformer Model Definition

In [ ]:
class TransformerDiacritizer(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, D_MODEL)
        self.pos_encoder = nn.Parameter(torch.zeros(1, MAX_LEN, D_MODEL))

        encoder_layers = nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=N_HEAD, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=N_LAYERS)
        self.fc_out = nn.Linear(D_MODEL, num_classes)

    def forward(self, x):
        x = self.embedding(x) + self.pos_encoder[:, :x.size(1), :]
        x = self.transformer_encoder(x)
        return self.fc_out(x)

# 4. Training Loop

In [ ]:
def train_model(model, train_loader, epochs=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=-1)

    model.train()
    for epoch in range(epochs):
        loop = tqdm(train_loader, leave=True)
        epoch_loss = 0
        for batch_idx, (inputs, targets) in enumerate(loop):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs.view(-1, len(DIACRITICS)), targets.view(-1))
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())
        print(f"Epoch {epoch+1} Avg Loss: {epoch_loss/len(train_loader):.4f}")

# 5. Execution

In [ ]:
# Adjust 'my_books/' to your folder name
# !mkdir my_books  # Uncomment to create a folder and upload your htm files there
FOLDER_PATH = "my_books/"

if os.path.exists(FOLDER_PATH):
    print("Extracting text...")
    raw_sentences = extract_raw_text(FOLDER_PATH)

    # Build Vocabulary
    all_text = "".join([araby.strip_tashkeel(s) for s in raw_sentences])
    unique_chars = sorted(list(set(all_text)))
    char_to_id = {c: i+1 for i, c in enumerate(unique_chars)}
    char_to_id['<PAD>'] = 0
    char_to_id['<UNK>'] = len(char_to_id)

    print(f"Dataset size: {len(raw_sentences)} sentences. Vocab size: {len(char_to_id)}")

    # Print samples of extracted text
    import random
    num_samples = min(5, len(raw_sentences))
    samples = random.sample(raw_sentences, num_samples)
    print(f"\n{'='*60}")
    print(f"Sample extracted text ({num_samples} sentences):")
    print(f"{'='*60}")
    for i, sent in enumerate(samples, 1):
        stripped = araby.strip_tashkeel(sent)
        print(f"\n--- Sample {i} ---")
        print(f"  Diacritized : {sent}")
        print(f"  Stripped    : {stripped}")
    print(f"\n{'='*60}\n")

    # Prepare Dataset
    dataset = DiacritizationDataset(raw_sentences, char_to_id)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

    # Initialize Model
    model = TransformerDiacritizer(len(char_to_id), len(DIACRITICS)).to(DEVICE)

    # Start Training
    train_model(model, loader, epochs=15)

    # Save the model and vocab for future use
    torch.save(model.state_dict(), "tashkeel_model.pth")
    with open("vocab.pkl", "wb") as f:
        pickle.dump(char_to_id, f)
    print("Model and vocabulary saved successfully.")
else:
    print(f"Folder '{FOLDER_PATH}' not found. Please upload your .htm files to a folder named 'my_books'.")

# 6. Inference Function

In [ ]:
def diacritize_text(model, text, char_to_id):
    model.eval()
    clean_text = araby.strip_tashkeel(text)
    input_ids = torch.tensor([[char_to_id.get(c, char_to_id['<UNK>']) for c in clean_text]]).to(DEVICE)

    with torch.no_grad():
        preds = model(input_ids)
        pred_classes = torch.argmax(preds, dim=-1).squeeze(0).cpu().numpy()

    result = ""
    for char, class_id in zip(clean_text, pred_classes):
        result += char + id_to_class[class_id]
    return result